In [ ]:
import os 
import wandb
import torch

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize
from pytorch_lightning.callbacks import ModelCheckpoint

from codefiles.helpers import set_all_seeds, build_model, build_lightningmodule, build_datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

def main(cfg) -> None:
    wandb.finish()
    set_all_seeds(seed=cfg.seed)
    wandb.init(
        project=cfg.wandb.project,
        group=None if cfg.wandb.group == "None" else cfg.wandb.group,
        config={key: value for key, value in cfg.items()},
    )
    
    checkpointaddon = ""
    if "corrupted_data_protocol" in cfg.modelname:
        if cfg.modelname.corrupted_data_protocol:
            checkpointaddon = "_corrupted"
        else:
            checkpointaddon = "_clean"

    checkpoint_name = f"debugmodel{checkpointaddon}"
    if os.path.exists(f'/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints/{checkpoint_name}.ckpt'):
        os.remove(f'/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints/{checkpoint_name}.ckpt')
    
    checkpoint_callback = ModelCheckpoint(
        monitor=cfg.encoders.monitor.metric, mode=cfg.encoders.monitor.mode,
        dirpath=f"/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints",
        filename=checkpoint_name,
        save_top_k=1,
    )

    model = build_model(cfg)
    lightningmodule = build_lightningmodule(cfg, model)
    datamodule = build_datamodule(cfg)

    trainer = pl.Trainer(
        logger=WandbLogger(project=cfg.wandb.project, dir="wandb/"),
        log_every_n_steps=1,
        accelerator='gpu',
        devices=1,
        max_epochs=cfg.max_epochs,
        precision=cfg.precision,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback] 
    )

    trainer.fit(lightningmodule, datamodule)
    trainer.test(ckpt_path='best', datamodule=datamodule)

    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")
    main(cfg)

Global seed set to 420
/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:269: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/loggers/wandb.py:395: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:165: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/l ...
GP

total_samples: 1284 / 1284
no_missing: 1284 / 1284
1_missing: 0 / 1284
2_missing: 0 / 1284
modality_0_missing: 0 / 1284
modality_1_missing: 0 / 1284
modality_2_missing: 0 / 1284
total_samples: 229 / 229
no_missing: 229 / 229
1_missing: 0 / 229
2_missing: 0 / 229
modality_0_missing: 0 / 229
modality_1_missing: 0 / 229
modality_2_missing: 0 / 229
total_samples: 686 / 686
no_missing: 686 / 686
1_missing: 0 / 686
2_missing: 0 / 686
modality_0_missing: 0 / 686
modality_1_missing: 0 / 686
modality_2_missing: 0 / 686


/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:613: Checkpoint directory /sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name        | Type                       | Params
------------------------------------------------------------
0  | model       | Multimodal_Architecture    | 283 M 
1  | loss        | CrossEntropyLoss           | 0     
2  | acc_2_train | BinaryAccuracy             | 0     
3  | acc_2_val   | BinaryAccuracy             | 0     
4  | acc_2_test  | BinaryAccuracy             | 0     
5  | acc_7_train | MulticlassAccuracy         | 0     
6  | acc_7_val   | MulticlassAccuracy         | 0     
7  | acc_7_test  | MulticlassAccuracy         | 0     
8  | f1_train    | BinaryF1Score              | 0     
9  | f1_val      | BinaryF1Score              | 0     
10 | f1_test     | BinaryF1Score        

Model GFlops (per instance): 14.77


Sanity Checking: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]